### link to part [PART 1 model training](https://www.kaggle.com/anantgupt/cassava-leaf-doctor-model-training-keras)

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
import glob

In [2]:
json_path = '../input/cassava-competition-trained-models/leaf-doctor-model-resnet50.json'
model_weights = '../input/cassava-competition-trained-models/leaf-doctor-wieghts-resnet50.hdf5'

with open(json_path, 'r') as json_file:
    json_savedModel = json_file.read()
    
model = tf.keras.models.model_from_json(json_savedModel)
model.load_weights(model_weights)

In [3]:
model.summary()

Model: "functional_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_3 (InputLayer)         [(None, 512, 512, 3)]     0         
_________________________________________________________________
sequential (Sequential)      (None, 512, 512, 3)       0         
_________________________________________________________________
resnet50 (Functional)        (None, 16, 16, 2048)      23587712  
_________________________________________________________________
global_average_pooling2d_1 ( (None, 2048)              0         
_________________________________________________________________
dense_2 (Dense)              (None, 512)               1049088   
_________________________________________________________________
batch_normalization (BatchNo (None, 512)               2048      
_________________________________________________________________
dropout_1 (Dropout)          (None, 512)              

In [4]:
test_images = glob.glob('../input/cassava-leaf-disease-classification/test_images/*.jpg')
print(test_images)

['../input/cassava-leaf-disease-classification/test_images/2216849948.jpg']


In [5]:
df = pd.DataFrame(np.array(test_images), columns=['Path'])
df

,Path
0,../input/cassava-leaf-disease-classification/t...


In [6]:
df = tf.data.Dataset.from_tensor_slices((df.Path.values))

row, col = 512,  512

def process(image_path):
    # load the raw data from the file as a string
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.random_brightness(img, 0.3)
    img = tf.image.random_flip_left_right(img, seed=None)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_crop(img, size=[row,col, 3])
    return img
    
df = df.map(process, num_parallel_calls=tf.data.experimental.AUTOTUNE).batch(8)

In [7]:
preds = []
for i in range(5):
    
    pred_test = model.predict(df, workers=16, verbose=1)
    preds.append(pred_test)

1/1 [==============================] - 0s 2ms/step


In [8]:
pred = np.mean(preds, axis=0)
pred

array([[0.00094334, 0.00251857, 0.41378117, 0.00163064, 0.5811263 ]],
      dtype=float32)

In [9]:
pred = np.argmax(pred, axis=-1)

In [10]:
sub = pd.read_csv('../input/cassava-leaf-disease-classification/sample_submission.csv')
sub['label'] = pred
sub

,image_id,label
0,2216849948.jpg,4


In [11]:
sub.to_csv('submission.csv', index=False)